In [4]:
import pandas as pd
import numpy as np
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier

In [5]:
X_train = joblib.load("../data/X_train_engineered.pkl")
X_val = joblib.load("../data/X_val_engineered.pkl")
X_test = joblib.load("../data/X_test_engineered.pkl")

y_train = joblib.load("../data/y_train.pkl")
y_val = joblib.load("../data/y_val.pkl")
y_test = joblib.load("../data/y_test.pkl")

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

X_train: (184506, 152)
X_val: (61502, 152)
X_test: (61503, 152)
y_train: (184506,)
y_val: (61502,)
y_test: (61503,)


In [6]:
numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))
print("Total features:", len(numerical_features) + len(categorical_features))

Numerical features: 136
Categorical features: 16
Total features: 152


In [7]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        drop="first"
    ))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [8]:
negative = (y_train == 0).sum()
positive = (y_train == 1).sum()

scale_pos_weight = negative / positive

print("Negative samples:", negative)
print("Positive samples:", positive)
print("Scale Pos Weight:", scale_pos_weight)

Negative samples: 169611
Positive samples: 14895
Scale Pos Weight: 11.38710976837865


In [9]:
xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric="auc",
        random_state=42,
        n_jobs=-1
    ))
])

In [10]:
xgb_model.fit(X_train, y_train)

print("XGBoost trained successfully!")

XGBoost trained successfully!


In [11]:
y_val_prob = xgb_model.predict_proba(X_val)[:, 1]

print("Validation predictions generated!")
print("Prediction shape:", y_val_prob.shape)

Validation predictions generated!
Prediction shape: (61502,)


In [12]:
from sklearn.metrics import roc_auc_score

val_auc = roc_auc_score(y_val, y_val_prob)

print("Validation ROC-AUC:", val_auc)

Validation ROC-AUC: 0.7669216913106711


In [13]:
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        solver="liblinear",
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)

print("Logistic Regression trained successfully!")

Logistic Regression trained successfully!


In [14]:
y_val_prob_lr = logistic_model.predict_proba(X_val)[:, 1]

lr_val_auc = roc_auc_score(y_val, y_val_prob_lr)

print("Logistic Regression Validation ROC-AUC:", lr_val_auc)

Logistic Regression Validation ROC-AUC: 0.6617620832428696


In [15]:
tuned_xgb = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=500,
        max_depth=5,
        learning_rate=0.03,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0.1,
        scale_pos_weight=scale_pos_weight,
        eval_metric="auc",
        random_state=42,
        n_jobs=-1
    ))
])

In [16]:
tuned_xgb.fit(X_train, y_train)

print("Tuned XGBoost trained successfully!")

Tuned XGBoost trained successfully!


In [17]:
y_val_prob_tuned = tuned_xgb.predict_proba(X_val)[:, 1]

tuned_val_auc = roc_auc_score(
    y_val,
    y_val_prob_tuned
)

print("Tuned XGBoost Validation ROC-AUC:", tuned_val_auc)

Tuned XGBoost Validation ROC-AUC: 0.7683697515699732


In [18]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

y_val_pred = (y_val_prob_tuned >= 0.5).astype(int)

print("Accuracy:", accuracy_score(y_val, y_val_pred))
print("Precision:", precision_score(y_val, y_val_pred))
print("Recall:", recall_score(y_val, y_val_pred))
print("F1 Score:", f1_score(y_val, y_val_pred))
print("ROC-AUC:", tuned_val_auc)

Accuracy: 0.7278462489024747
Precision: 0.1792971942250068
Recall: 0.6628398791540786
F1 Score: 0.28224699828473415
ROC-AUC: 0.7683697515699732


In [19]:
thresholds = np.arange(0.10, 0.91, 0.05)

threshold_results = []

for threshold in thresholds:
    pred = (y_val_prob_tuned >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_val, pred),
        "Recall": recall_score(y_val, pred),
        "F1": f1_score(y_val, pred)
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df

,Threshold,Precision,Recall,F1
0,0.10,0.084242,0.995569,0.155340
1,0.15,0.090639,0.983283,0.165978
2,0.20,0.099527,0.962135,0.180393
3,0.25,0.109313,0.930312,0.195637
4,0.30,0.120477,0.890634,0.212244
5,0.35,0.132608,0.844713,0.229230
6,0.40,0.146106,0.790534,0.246630
7,0.45,0.161414,0.727291,0.264194
8,0.50,0.179297,0.662840,0.282247
9,0.55,0.198912,0.588922,0.297381


In [20]:
best_threshold = threshold_df.loc[
    threshold_df["F1"].idxmax(),
    "Threshold"
]

best_f1 = threshold_df["F1"].max()

print("Best validation threshold:", best_threshold)
print("Best validation F1:", best_f1)

Best validation threshold: 0.6500000000000001
Best validation F1: 0.3127347720409516


In [21]:
joblib.dump(
    tuned_xgb,
    "../data/final_xgb_model.pkl"
)

joblib.dump(
    best_threshold,
    "../data/best_threshold.pkl"
)

print("Final model saved successfully!")
print("Final threshold:", best_threshold)

Final model saved successfully!
Final threshold: 0.6500000000000001


In [22]:
validation_results = {
    "model": "Tuned XGBoost",
    "validation_auc": tuned_val_auc,
    "threshold": best_threshold,
    "validation_f1": best_f1
}

joblib.dump(
    validation_results,
    "../data/validation_results.pkl"
)

print("Validation results saved!")

Validation results saved!


# Model Selection Summary

Three models were evaluated using the validation dataset. Logistic Regression achieved a ROC-AUC of approximately 0.662, while the baseline XGBoost model achieved approximately 0.767.

The tuned XGBoost model achieved the highest validation ROC-AUC of approximately 0.768 and was therefore selected as the final predictive model.

The classification threshold was optimized using the validation dataset. A threshold of 0.65 produced the highest F1-score among the evaluated thresholds, achieving an F1-score of approximately 0.313, precision of approximately 0.247, and recall of approximately 0.428.

The model and selected threshold were then frozen before evaluation on the independent test dataset.

In [23]:
import lightgbm as lgb

print("LightGBM version:", lgb.__version__)

LightGBM version: 4.7.0


In [26]:
X_train_lgb = X_train.copy()
X_val_lgb = X_val.copy()

categorical_features_lgb = X_train_lgb.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

for col in categorical_features_lgb:
    X_train_lgb[col] = X_train_lgb[col].astype("category")
    X_val_lgb[col] = X_val_lgb[col].astype("category")

print("Categorical features:", len(categorical_features_lgb))
print("Train shape:", X_train_lgb.shape)
print("Validation shape:", X_val_lgb.shape)

Categorical features: 16
Train shape: (184506, 152)
Validation shape: (61502, 152)


In [27]:
lgb_model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

print("LightGBM model created!")

LightGBM model created!


In [28]:
lgb_model.fit(
    X_train_lgb,
    y_train,
    categorical_feature=categorical_features_lgb
)

print("LightGBM trained successfully!")

LightGBM trained successfully!


In [29]:
y_val_prob_lgb = lgb_model.predict_proba(X_val_lgb)[:, 1]

lgb_val_auc = roc_auc_score(
    y_val,
    y_val_prob_lgb
)

print("LightGBM Validation ROC-AUC:", lgb_val_auc)

LightGBM Validation ROC-AUC: 0.7685444181755797


In [30]:
X_train_base = X_train_lgb.iloc[:, :121].copy()
X_val_base = X_val_lgb.iloc[:, :121].copy()

categorical_base = X_train_base.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

for col in categorical_base:
    X_train_base[col] = X_train_base[col].astype("category")
    X_val_base[col] = X_val_base[col].astype("category")

lgb_base = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgb_base.fit(
    X_train_base,
    y_train,
    categorical_feature=categorical_base
)

y_val_prob_base = lgb_base.predict_proba(X_val_base)[:, 1]

base_auc = roc_auc_score(y_val, y_val_prob_base)

print("Original 121 Features - LightGBM ROC-AUC:", base_auc)
print("Engineered 152 Features - LightGBM ROC-AUC:", lgb_val_auc)

Original 121 Features - LightGBM ROC-AUC: 0.7550192807458602
Engineered 152 Features - LightGBM ROC-AUC: 0.7685444181755797


In [31]:
def add_risk_features(df):
    df = df.copy()

    # Bureau utilization
    df["BUREAU_DEBT_CREDIT_RATIO"] = (
        df["BUREAU_DEBT_SUM"] /
        df["BUREAU_CREDIT_SUM"].replace(0, np.nan)
    )

    # Bureau active-loan ratio
    df["BUREAU_ACTIVE_RATIO"] = (
        df["BUREAU_ACTIVE_COUNT"] /
        df["BUREAU_LOAN_COUNT"].replace(0, np.nan)
    )

    # Bureau overdue ratio
    df["BUREAU_OVERDUE_CREDIT_RATIO"] = (
        df["BUREAU_OVERDUE_SUM"] /
        df["BUREAU_CREDIT_SUM"].replace(0, np.nan)
    )

    # Previous application approval rate
    df["PREV_APPROVAL_RATE"] = (
        df["PREV_APPROVED_COUNT"] /
        df["PREV_APP_COUNT"].replace(0, np.nan)
    )

    # Previous application refusal rate
    df["PREV_REFUSAL_RATE"] = (
        df["PREV_REFUSED_COUNT"] /
        df["PREV_APP_COUNT"].replace(0, np.nan)
    )

    # Installment payment coverage
    df["INSTALLMENT_PAYMENT_COVERAGE"] = (
        df["INSTALLMENT_PAYMENT_SUM"] /
        df["INSTALLMENT_INSTALMENT_SUM"].replace(0, np.nan)
    )

    # Debt relative to income
    df["BUREAU_DEBT_INCOME_RATIO"] = (
        df["BUREAU_DEBT_SUM"] /
        df["AMT_INCOME_TOTAL"].replace(0, np.nan)
    )

    return df

In [32]:
X_train_risk = add_risk_features(X_train_lgb)
X_val_risk = add_risk_features(X_val_lgb)

print("Original features:", X_train_lgb.shape[1])
print("Risk-engineered features:", X_train_risk.shape[1])

Original features: 152
Risk-engineered features: 159


In [33]:
X_train_risk = add_risk_features(X_train_lgb)
X_val_risk = add_risk_features(X_val_lgb)

print("Train shape:", X_train_risk.shape)
print("Validation shape:", X_val_risk.shape)

Train shape: (184506, 159)
Validation shape: (61502, 159)


In [34]:
categorical_risk = X_train_risk.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

for col in categorical_risk:
    X_train_risk[col] = X_train_risk[col].astype("category")
    X_val_risk[col] = X_val_risk[col].astype("category")

print("Categorical features:", len(categorical_risk))

Categorical features: 16


In [36]:
lgb_risk = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgb_risk.fit(
    X_train_risk,
    y_train,
    categorical_feature=categorical_risk
)

print("Risk-feature LightGBM trained successfully!")

Risk-feature LightGBM trained successfully!


In [37]:
y_val_prob_risk = lgb_risk.predict_proba(
    X_val_risk
)[:, 1]

risk_auc = roc_auc_score(
    y_val,
    y_val_prob_risk
)

print("Risk-feature LightGBM Validation ROC-AUC:", risk_auc)
print("Previous best:", lgb_val_auc)

Risk-feature LightGBM Validation ROC-AUC: 0.7694745864274715
Previous best: 0.7685444181755797


In [38]:
lgb_tuned = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=1000,
    learning_rate=0.02,

    num_leaves=63,
    max_depth=-1,
    min_child_samples=100,

    subsample=0.8,
    colsample_bytree=0.8,

    reg_alpha=0.5,
    reg_lambda=1.0,

    scale_pos_weight=scale_pos_weight,

    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

print("Tuned LightGBM created!")

Tuned LightGBM created!


In [39]:
lgb_tuned.fit(
    X_train_risk,
    y_train,
    categorical_feature=categorical_risk
)

print("Tuned LightGBM trained successfully!")

Tuned LightGBM trained successfully!


In [40]:
y_val_prob_lgb_tuned = lgb_tuned.predict_proba(
    X_val_risk
)[:, 1]

tuned_lgb_auc = roc_auc_score(
    y_val,
    y_val_prob_lgb_tuned
)

print("Tuned LightGBM Validation ROC-AUC:", tuned_lgb_auc)
print("Current best:", risk_auc)

Tuned LightGBM Validation ROC-AUC: 0.7679411468656347
Current best: 0.7694745864274715
